# PDF Document ingestion and chunking using Data Prep Kit

This notebook will introduce two DPK transforms that are based on docling. 

Here is the workflow:

- docling2parquet: Extract text from PDF documents
- doc_chunk: Break document into chunks

<b>Pre-requisite</b>: You need to download one or more PDF files for testing. The current release of the notebook was tested with [this file](https://github.com/user-attachments/files/20354534/opea_project_github_io_latest_introduction_index_html.pdf) provided by @ezelanza in [this issue](https://github.com/data-prep-kit/data-prep-kit/issues/1288). We assume the test PDF(s) is/are downloaded into the notebook folder. 
  

## How to run this notebook

If you have python 3.11 or higher on your machine, you can also download the notebook and run it locally using a local python environment setup as follows:

```
python -m venv venv
source venv/bin/activate
pip install jupyterlab
jupyter lab docling.ipynb
```

For more advanced setup, please see setup [guide](https://github.com/data-prep-kit/data-prep-kit/blob/dev/doc/quick-start/quick-start.md).

An earlier version of this notebook was tested successfully on Google Colab. However, continuous changes in the Colab environment could introduce unexpected behavior/breakage. If you wish to try the Colab environment, click on [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data-prep-kit/data-prep-kit/blob/dev/recipes/DPK-sequence/docling.ipynb). 


## Step-1 Install DPK and a couple of helper applictions

In [ ]:
#%%capture cap --no-stderr
#%pip install "data-prep-toolkit-transforms[docling2parquet,doc_chunk]"
%pip install "data-prep-toolkit @ git+https://github.com/touma-I/data-prep-kit-pkg@numpy-dependencies#subdirectory=data-processing-lib"
%pip install "data-prep-toolkit-transforms[docling2parquet,doc_chunk] @ git+https://github.com/touma-I/data-prep-kit-pkg@numpy-dependencies#subdirectory=transforms"


## Step-2: Extract Data from PDF (docling2parquet)

This step we will read PDF files and extract the text data.

[Docling2Parquet documentation](https://github.com/data-prep-kit/data-prep-kit/blob/dev/transforms/language/docling2parquet/README.md)

We use the [Docling package](https://github.com/DS4SD/docling).


In [ ]:
%%time

from dpk_docling2parquet import Docling2Parquet
from dpk_docling2parquet import docling2parquet_contents_types

result = Docling2Parquet(input_folder= ".",
                    output_folder= "docling2parquet",
                    data_files_to_use=['.pdf'],
                    docling2parquet_contents_type=docling2parquet_contents_types.MARKDOWN,   # markdown
                    ).transform()


### 2.1: Inspect Generated parquet

Here we should see one entry per input file processed.

In [ ]:
import pandas as pd
import glob

df = pd.concat(
    (pd.read_parquet(parquet_file)
    for parquet_file in glob.glob("docling2parquet/*.parquet")),
    ignore_index=True
)
df

## Step-3: Breakup document into chunks


[doc_chunk Documentation](https://github.com/data-prep-kit/data-prep-kit/blob/dev/transforms/language/doc_chunk/README.md)


In [ ]:
%%time

from dpk_doc_chunk import DocChunk

result = DocChunk(input_folder="docling2parquet",
        output_folder="doc_chunk",
        doc_chunk_chunking_type= "li_markdown",
        doc_chunk_chunk_size_tokens = 128,  # default 128
        doc_chunk_chunk_overlap_tokens=30   # default 30
        ).transform()

## 3.1: Inspect Generated parquet

Here we should see one entry per input file processed.

In [ ]:
import pandas as pd
import glob

df = pd.concat(
    (pd.read_parquet(parquet_file)
    for parquet_file in glob.glob("doc_chunk/*.parquet")),
    ignore_index=True
)
df